# ETHICS Regional-Cultural Persona — Analisis Akurasi v4 (Region-First)

Notebook ini mengikuti urutan penyajian notebook Gender dan Religion:
perbandingan langsung antarkawasan sebagai hasil utama, kemudian skenario
implisit–eksplisit, persona–netral, serta status respons. Empat subset menjadi
konteks analisis dan tidak digabung menjadi satu angka akurasi ETHICS.

## 0. Provenance dan validasi

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

V3_DIR = Path(r"D:\TugasAkhir\result\ETHICS\reports\Regional_Cultural_Persona\_combined\persona_publication_analysis_v3_accuracy_paired_holm_six_models")
THESIS_ROOT = Path(r"D:\ProposalTA-18222008")
TABLE_DIR = THESIS_ROOT / "analisis" / "ethics_accuracy" / "regional"
FIGURE_DIR = THESIS_ROOT / "image" / "BabVI" / "ethics" / "regional_kultural"

manifest = json.loads((V3_DIR / "metadata" / "analysis_manifest.json").read_text(encoding="utf-8"))
validation = json.loads((V3_DIR / "metadata" / "validation.json").read_text(encoding="utf-8"))
assert manifest["analysis_version"] == "ethics-regional-v3-accuracy-paired-holm-20260822"
assert validation["valid"] and validation["rows_processed"] == 154428

model_summary = pd.read_csv(TABLE_DIR / "summary_by_model.csv")
identity_tendency = pd.read_csv(TABLE_DIR / "identity_tendency.csv")
status_model = pd.read_csv(TABLE_DIR / "response_status_by_model.csv")
status_subset = pd.read_csv(TABLE_DIR / "response_status_by_subset.csv")
status_detail = pd.read_csv(TABLE_DIR / "response_status_by_subset_model_persona.csv")
reason_detail = pd.read_csv(TABLE_DIR / "abstain_reason_by_subset_model_persona.csv")
display(pd.DataFrame({"value": [manifest["analysis_version"], validation["rows_processed"], validation["valid"]]}, index=["analysis_version", "rows_processed", "valid"]))

## 1. Perbandingan antarkawasan — hasil utama

Dari 576 perbandingan langsung pada soal dan skenario yang sama, 115 tetap
signifikan setelah koreksi Holm. Hasil muncul pada keenam model, tetapi arahnya
tidak seragam. Di antara hasil signifikan, Amerika Utara dan Asia Timur menjadi
sisi dengan akurasi lebih tinggi pada masing-masing 36 hasil, Asia Selatan pada
25 hasil, dan Afrika Sub-Sahara pada 18 hasil. Hitungan ini tidak membentuk
urutan akurasi kawasan secara umum.

In [ ]:
display(model_summary[["model", "direct_significant", "direct_dominant_higher", "direct_scenarios", "direct_largest_comparison", "direct_largest_subset", "direct_largest_delta_pp", "direct_largest_h"]].style.format({"direct_largest_delta_pp": "{:+.2f}", "direct_largest_h": "{:+.2f}"}))
display(identity_tendency)
display(Image(filename=str(FIGURE_DIR / "01_regional_direct_effect_holm_heatmap.png"), width=1500))

## 2. Perbandingan skenario implisit dan eksplisit

Sebanyak 139 dari 288 perbandingan I1–I3 terhadap E didukung Q Cochran dan
koreksi Holm. Akurasi skenario implisit lebih tinggi pada 30 hasil dan lebih
rendah pada 109 hasil. I1, I2, dan I3 tetap diuji secara terpisah.

In [ ]:
display(model_summary[["model", "ie_significant", "ie_higher", "ie_lower"]])
display(Image(filename=str(FIGURE_DIR / "02_regional_implicit_vs_explicit_effect_q_holm_heatmap.png"), width=1500))

## 3. Perbandingan skenario persona dan netral

Sebanyak 145 dari 384 perbandingan E dan I1–I3 terhadap N didukung Q Cochran
dan koreksi Holm. Akurasi skenario persona lebih tinggi pada 51 hasil dan lebih
rendah pada 94 hasil.

In [ ]:
display(model_summary[["model", "neutral_significant", "neutral_higher", "neutral_lower"]])
display(Image(filename=str(FIGURE_DIR / "03_regional_persona_vs_neutral_effect_q_holm_heatmap.png"), width=1500))

## 4. Akurasi dan status respons

Akurasi penuh dibaca bersama commit rate, abstain, missing, dan akurasi pada
respons committed. Pada 109 dari 115 perbandingan antarkawasan signifikan,
sisi dengan akurasi penuh lebih tinggi juga memiliki abstain lebih rendah.

In [ ]:
MODEL_LABELS = {
    "qwen_qwen3-next-80b-a3b-thinking": "Qwen3 Next 80B-A3B",
    "meta-llama_llama-3.1-8b-instruct": "Llama 3.1 8B",
    "mistralai_ministral-8b-2512": "Ministral 8B",
    "deepseek_deepseek-r1-distill-qwen-32b": "DeepSeek R1-Distill 32B",
    "openai_gpt-oss-20b": "GPT-OSS 20B",
    "meta-llama_llama-3.3-70b-instruct": "Llama 3.3 70B",
}
status_model_view = status_model.assign(model_label=status_model["model"].map(MODEL_LABELS))
display(status_model_view[["model_label", "accuracy_full_pct", "committed_pct", "abstain_pct", "missing_pct", "accuracy_committed_pct"]].style.format(precision=2))
display(status_subset.style.format(precision=2))

### 4.1 Kondisi yang rentan tidak memberi keputusan pada Justice

Rata-rata abstain tertinggi pada Justice terdapat pada DeepSeek R1-Distill 32B
dan Llama 3.1 8B. Pada DeepSeek, Afrika Sub-Sahara memiliki rata-rata abstain
tertinggi; pada Llama, Afrika Sub-Sahara dan Asia Timur menjadi dua kondisi
tertinggi. Pola tersebut dibaca sebagai interaksi model–persona–skenario.

In [ ]:
justice = status_detail[(status_detail["subset"] == "justice") & (status_detail["persona_folder"] != "neutral")].copy()
justice["identity"] = justice["persona_folder"].str.replace(r"_(explicit|implicit_[123])$", "", regex=True)
justice_average = (justice.groupby(["model", "identity"], as_index=False)
    .agg(abstain_pct=("abstain_pct", "mean"), accuracy_full_pct=("accuracy_full_pct", "mean")))
justice_average["model_label"] = justice_average["model"].map(MODEL_LABELS)
display(justice_average.sort_values(["model_label", "abstain_pct"], ascending=[True, False]).style.format({"abstain_pct": "{:.2f}", "accuracy_full_pct": "{:.2f}"}))

focus = reason_detail[(reason_detail["subset"] == "justice") & (reason_detail["persona_folder"].str.endswith("implicit_1"))]
focus = focus.sort_values(["model", "persona_folder", "n_abstain_reason"], ascending=[True, True, False]).groupby(["model", "persona_folder"]).head(3)
display(focus[["model", "persona_folder", "abstain_reason", "n_abstain_reason", "n_abstain_total", "percentage_within_condition_abstain"]].style.format({"percentage_within_condition_abstain": "{:.2f}"}))

## 5. Batas interpretasi subset dan detail lengkap

Inferensi tetap dilakukan per subset. Commonsense dan Justice dapat dibaca
sebagai ketepatan keputusan biner. Deontology kehilangan sebagian informasi
penentu label, sedangkan Virtue memiliki ketidaksesuaian antara pertanyaan dan
arti label. Seluruh kombinasi per subset disediakan pada lampiran tesis;
notebook ini tidak menggunakan satu angka akurasi gabungan empat subset.

## 6. Catatan reproduksi

Notebook ini adalah lapisan penyajian. Pengujian statistik tetap berasal dari
`regional_publication_analysis_v3_accuracy.py` dan bundle v3 tervalidasi.
Grafik diperbesar dengan mengubah rasio kanvas serta font judul, legenda,
sumbu x, sumbu y, anotasi, dan colorbar pada
`scripts/build_ethics_accuracy_religion_regional.py`.